# Notebook 04: Calculate TPM

Compute Transcripts Per Million (TPM) from raw read counts and gene lengths.

**Critical: TPM must be calculated on the full ~14,690 gene set BEFORE the inner join
with FANTOM5 (which reduces to 12,544 genes).** The scaling factor denominator changes
with gene count.

**Critical Test #2:** Calculate TPM both ways (before and after join) to quantify
the impact of this methodological decision.

**Formula:**
```
RPK_i = count_i / (gene_length_bp_i / 1000)
scaling_factor = sum(RPK) / 1e6
TPM_i = RPK_i / scaling_factor
```

**Pseudocount:** 0.01 for Log2FC to handle zeros.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
REFERENCE_DIR = Path("../reference")

## 1. Load Data

In [2]:
# Load Hubstenberger data with gene lengths (from notebook 03)
hub = pd.read_csv(PROCESSED_DIR / "03_hub_with_gene_lengths.csv")
print(f"Loaded {len(hub):,} protein-coding genes with gene lengths")

# Identify the raw read count columns
pbody_count_cols = [c for c in hub.columns if 'sorted P-body' in c and 'mapped read number' in c]
cytosol_count_cols = [c for c in hub.columns if 'pre-sorted fraction' in c and 'mapped read number' in c]

print(f"\nP-body count columns ({len(pbody_count_cols)}):")
for c in pbody_count_cols:
    print(f"  {c}")
print(f"\nCytosol count columns ({len(cytosol_count_cols)}):")
for c in cytosol_count_cols:
    print(f"  {c}")

all_count_cols = pbody_count_cols + cytosol_count_cols
assert len(all_count_cols) == 6, f"Expected 6 count columns, got {len(all_count_cols)}"

Loaded 15,281 protein-coding genes with gene lengths

P-body count columns (3):
  sorted P-body replicate 1 (mapped read number)
  sorted P-body replicate 2 (mapped read number)
  sorted P-body replicate 3 (mapped read number)

Cytosol count columns (3):
  pre-sorted fraction replicate 1 (mapped read number)
  pre-sorted fraction replicate 2 (mapped read number)
  pre-sorted fraction replicate 3 (mapped read number)


## 2. Calculate TPM (Full Gene Set)

Vectorized TPM calculation on the full ~14,690 protein-coding gene set.

In [3]:
def compute_tpm(counts_df, gene_lengths_bp, sample_columns):
    """Vectorized TPM calculation.
    
    TPM = (count / gene_length_kb) / sum(count / gene_length_kb) * 1e6
    """
    gene_lengths_kb = gene_lengths_bp / 1000
    rpk = counts_df[sample_columns].div(gene_lengths_kb, axis=0)
    scaling_factors = rpk.sum(axis=0) / 1e6
    tpm = rpk.div(scaling_factors, axis=1)
    return tpm


# Calculate TPM on full gene set (CORRECT method — before join)
tpm_full = compute_tpm(hub, hub["gene_length_bp"], all_count_cols)

# Rename columns
tpm_names = [
    "PB_rep1_TPM", "PB_rep2_TPM", "PB_rep3_TPM",
    "Cytosol_rep1_TPM", "Cytosol_rep2_TPM", "Cytosol_rep3_TPM"
]
tpm_full.columns = tpm_names

# Add to hub dataframe
for col in tpm_names:
    hub[col] = tpm_full[col].values

# Calculate averages
hub["PB_TPM"] = hub[["PB_rep1_TPM", "PB_rep2_TPM", "PB_rep3_TPM"]].mean(axis=1)
hub["Cytosol_TPM"] = hub[["Cytosol_rep1_TPM", "Cytosol_rep2_TPM", "Cytosol_rep3_TPM"]].mean(axis=1)

# Calculate Log2FC with pseudocount
PSEUDOCOUNT = 0.01
hub["NEW_Log2FC_TPM"] = np.log2(
    (hub["PB_TPM"] + PSEUDOCOUNT) / (hub["Cytosol_TPM"] + PSEUDOCOUNT)
)

print(f"TPM calculated for {len(hub):,} genes across 6 replicates")
print(f"\nTPM summary (Cytosol_TPM):")
print(hub["Cytosol_TPM"].describe())
print(f"\nNEW_Log2FC_TPM summary:")
print(hub["NEW_Log2FC_TPM"].describe())

TPM calculated for 15,281 genes across 6 replicates

TPM summary (Cytosol_TPM):
count    15281.000000
mean        65.440743
std        615.977860
min          0.000000
25%          2.867609
50%         17.112202
75%         54.265446
max      48733.255650
Name: Cytosol_TPM, dtype: float64

NEW_Log2FC_TPM summary:
count    15281.000000
mean        -1.116561
std          2.028249
min         -9.697707
25%         -2.612015
50%         -1.000869
75%          0.564666
max          6.695785
Name: NEW_Log2FC_TPM, dtype: float64


## 3. Critical Test: TPM Before vs After Join

Jason flagged this as a correction: TPM should be calculated on the full protein-coding
set (~14,690) not the post-join set (12,544). Quantify the actual difference.

In [4]:
# Load reference to get the 12,544 post-join gene IDs
ref = pd.read_csv(REFERENCE_DIR / "12544_Hub_CAGE_MERGE_with_CapIndex.csv")
post_join_ids = set(ref["ensembl_id"].unique())

# Subset hub to post-join genes
hub_post = hub[hub["ensembl_id"].isin(post_join_ids)].copy()
print(f"Post-join gene count: {len(hub_post):,}")

# Calculate TPM on the REDUCED set (INCORRECT method — after join)
tpm_reduced = compute_tpm(hub_post, hub_post["gene_length_bp"], all_count_cols)
tpm_reduced.columns = [f"{c}_afterjoin" for c in tpm_names]

# Compare the two TPM calculations for the overlapping 12,544 genes
tpm_before = hub[hub["ensembl_id"].isin(post_join_ids)][tpm_names].reset_index(drop=True)
tpm_after = tpm_reduced.reset_index(drop=True)
tpm_after.columns = tpm_names  # rename back for comparison

print("\n=== TPM Before vs After Join ===")
for col in tpm_names:
    r, _ = stats.pearsonr(tpm_before[col], tpm_after[col])
    max_diff = (tpm_before[col] - tpm_after[col]).abs().max()
    mean_diff = (tpm_before[col] - tpm_after[col]).abs().mean()
    print(f"  {col}: r={r:.10f}, max_diff={max_diff:.6f}, mean_diff={mean_diff:.6f}")

# Check averages
pb_before = tpm_before[["PB_rep1_TPM", "PB_rep2_TPM", "PB_rep3_TPM"]].mean(axis=1)
pb_after = tpm_after[["PB_rep1_TPM", "PB_rep2_TPM", "PB_rep3_TPM"]].mean(axis=1)
cyt_before = tpm_before[["Cytosol_rep1_TPM", "Cytosol_rep2_TPM", "Cytosol_rep3_TPM"]].mean(axis=1)
cyt_after = tpm_after[["Cytosol_rep1_TPM", "Cytosol_rep2_TPM", "Cytosol_rep3_TPM"]].mean(axis=1)

log2fc_before = np.log2((pb_before + PSEUDOCOUNT) / (cyt_before + PSEUDOCOUNT))
log2fc_after = np.log2((pb_after + PSEUDOCOUNT) / (cyt_after + PSEUDOCOUNT))

# Sign flips in Log2FC?
sign_flips = ((log2fc_before > 0) != (log2fc_after > 0)).sum()
print(f"\n  Log2FC sign flips: {sign_flips} / {len(log2fc_before):,} genes")
print(f"  Log2FC Pearson r: {stats.pearsonr(log2fc_before, log2fc_after)[0]:.10f}")
print(f"  Log2FC max diff: {(log2fc_before - log2fc_after).abs().max():.6f}")

Post-join gene count: 12,510

=== TPM Before vs After Join ===
  PB_rep1_TPM: r=1.0000000000, max_diff=673.695116, mean_diff=9.434924
  PB_rep2_TPM: r=1.0000000000, max_diff=1082.683237, mean_diff=9.322553
  PB_rep3_TPM: r=1.0000000000, max_diff=911.237612, mean_diff=9.611985
  Cytosol_rep1_TPM: r=1.0000000000, max_diff=2722.190022, mean_diff=25.612261
  Cytosol_rep2_TPM: r=1.0000000000, max_diff=1770.323961, mean_diff=21.712904
  Cytosol_rep3_TPM: r=1.0000000000, max_diff=2423.848244, mean_diff=26.615119

  Log2FC sign flips: 764 / 12,510 genes
  Log2FC Pearson r: 0.9999370961
  Log2FC max diff: 0.578673


## 4. Validation Against Jason's Reference

Compare our TPM values (calculated on full gene set) against Jason's reference
for the 12,544 overlapping genes.

In [5]:
# Merge our data with reference on ensembl_id
our_cols = ["ensembl_id"] + tpm_names + ["PB_TPM", "Cytosol_TPM", "NEW_Log2FC_TPM"]
ours = hub[hub["ensembl_id"].isin(post_join_ids)][our_cols].copy()

ref_cols = ["ensembl_id"] + tpm_names + ["PB_TPM", "Cytosol_TPM", "NEW_Log2FC_TPM"]
ref_subset = ref[ref_cols].copy()

comp = ours.merge(ref_subset, on="ensembl_id", suffixes=("_ours", "_ref"))
print(f"Comparing {len(comp):,} genes\n")

# Compare each column
check_cols = tpm_names + ["PB_TPM", "Cytosol_TPM", "NEW_Log2FC_TPM"]
print(f"{'Column':<25} {'Pearson r':>12} {'Max diff':>12} {'Close (1e-5)':>14}")
print("-" * 65)

for col in check_cols:
    ours_vals = comp[f"{col}_ours"]
    ref_vals = comp[f"{col}_ref"]
    r, _ = stats.pearsonr(ours_vals, ref_vals)
    max_diff = (ours_vals - ref_vals).abs().max()
    close = np.allclose(ours_vals, ref_vals, rtol=1e-5, atol=1e-8, equal_nan=True)
    print(f"{col:<25} {r:>12.10f} {max_diff:>12.6f} {'PASS' if close else 'FAIL':>14}")

Comparing 12,510 genes

Column                       Pearson r     Max diff   Close (1e-5)
-----------------------------------------------------------------
PB_rep1_TPM               1.0000000000    67.518313           FAIL
PB_rep2_TPM               1.0000000000   103.063400           FAIL
PB_rep3_TPM               1.0000000000    90.071609           FAIL
Cytosol_rep1_TPM          1.0000000000   101.646356           FAIL
Cytosol_rep2_TPM          1.0000000000    81.692962           FAIL
Cytosol_rep3_TPM          1.0000000000    90.185346           FAIL
PB_TPM                    0.9999999991    86.884441           FAIL
Cytosol_TPM               0.9999999985    91.174887           FAIL
NEW_Log2FC_TPM            0.9999996617     0.026249           FAIL


## 5. Save Output

In [6]:
output_path = PROCESSED_DIR / "04_hub_with_tpm.csv"
hub.to_csv(output_path, index=False)
print(f"Saved {len(hub):,} genes with TPM to {output_path}")
print(f"Columns: {len(hub.columns)}")

Saved 15,281 genes with TPM to ../data/processed/04_hub_with_tpm.csv
Columns: 32
